# SupportOps AI - DistilBERT Intent Classification

## Objective

Fine-tune a pretrained DistilBERT Transformer model for
77-class customer-support intent classification using BANKING77.

## Model Comparison

Classical Baseline:
TF-IDF + Logistic Regression

Transformer:
DistilBERT

## Evaluation

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Per-intent F1
- Error Analysis
- Official BANKING77 Test Set
- Strict Zero-Overlap Test Set

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Libraries imported successfully!")

Check PyTorch device inside notebook

In [ ]:
print("PyTorch version:", torch.__version__)

print(
    "MPS available:",
    torch.backends.mps.is_available()
)

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Load BANKING77

In [ ]:
TRAIN_PATH = "../data/raw/banking77_train.csv"
TEST_PATH = "../data/raw/banking77_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df = train_df[
    ["text", "intent"]
].copy()

test_df = test_df[
    ["text", "intent"]
].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

train_df.head()

Create train/validation sets

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["intent"]
)

print("Training subset:", len(train_data))
print("Validation subset:", len(val_data))
print("Official test:", len(test_df))

Convert labels into numbers

In [ ]:
intent_names = sorted(
    train_df["intent"].unique()
)

print("Number of intents:", len(intent_names))

In [ ]:
#Create mappings
label2id = {
    intent: idx
    for idx, intent in enumerate(intent_names)
}

id2label = {
    idx: intent
    for intent, idx in label2id.items()
}

In [ ]:
list(label2id.items())[:10]

Add numeric labels

In [ ]:
train_data = train_data.copy()
val_data = val_data.copy()
test_data = test_df.copy()

train_data["label"] = (
    train_data["intent"].map(label2id)
)

val_data["label"] = (
    val_data["intent"].map(label2id)
)

test_data["label"] = (
    test_data["intent"].map(label2id)
)

In [ ]:
train_data[
    ["text", "intent", "label"]
].head()

Load the tokenizer

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully!")

See tokenization

In [ ]:
sample_text = (
    "I was charged twice for the same card payment."
)

tokens = tokenizer.tokenize(
    sample_text
)

print(tokens)

In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(
    tokens
)

print(token_ids)

Special tokens

In [ ]:
encoded = tokenizer(
    sample_text
)

encoded

Decode the Id's

In [ ]:
tokenizer.convert_ids_to_tokens(
    encoded["input_ids"]
)

In [ ]:
token_lengths = []

for text in train_df["text"]:
    
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )
    
    token_lengths.append(
        len(encoded["input_ids"])
    )

In [ ]:
token_length_series = pd.Series(
    token_lengths
)

token_length_series.describe(
    percentiles=[
        0.90,
        0.95,
        0.99
    ]
)

Plot token lengths

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    token_lengths,
    bins=30
)

plt.title(
    "BANKING77 DistilBERT Token Length Distribution"
)

plt.xlabel(
    "Number of Tokens"
)

plt.ylabel(
    "Number of Queries"
)

plt.tight_layout()
plt.show()

Tokenize the datasets

In [ ]:
MAX_LENGTH = 64

In [ ]:
sample_encoding = tokenizer(
    "I was charged twice for the same card payment.",
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

sample_encoding

In [ ]:
print(
    "Input IDs shape:",
    sample_encoding["input_ids"].shape
)

print(
    "Attention mask shape:",
    sample_encoding["attention_mask"].shape
)

Understand padding

In [ ]:
print(
    tokenizer.convert_ids_to_tokens(
        sample_encoding["input_ids"][0]
    )
)

In [ ]:
print(
    sample_encoding["attention_mask"][0]
)

Create a PyTorch Dataset

In [ ]:
from torch.utils.data import Dataset

In [ ]:
class Banking77Dataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )
        }

Create train, validation and test datasets

In [ ]:
train_dataset = Banking77Dataset(
    train_data["text"],
    train_data["label"],
    tokenizer,
    MAX_LENGTH
)

val_dataset = Banking77Dataset(
    val_data["text"],
    val_data["label"],
    tokenizer,
    MAX_LENGTH
)

test_dataset = Banking77Dataset(
    test_data["text"],
    test_data["label"],
    tokenizer,
    MAX_LENGTH
)

In [ ]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Inspect one encoded example

In [ ]:
example = train_dataset[0]

print(
    "Input IDs shape:",
    example["input_ids"].shape
)

print(
    "Attention mask shape:",
    example["attention_mask"].shape
)

print(
    "Label:",
    example["labels"]
)

In [ ]:
print(
    tokenizer.decode(
        example["input_ids"],
        skip_special_tokens=True
    )
)

In [ ]:
print(
    "Intent:",
    id2label[
        example["labels"].item()
    ]
)

Load pretrained DistilBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=77,
    id2label=id2label,
    label2id=label2id
)

Move model to your device

In [ ]:
model = model.to(device)

print(
    "Model loaded on:",
    device
)

Import Trainer components

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments
)

print("Trainer components imported!")

Create the evaluation function

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Some transformer models can return predictions as tuples
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

Training configuration

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
training_args = TrainingArguments(

    # Where checkpoints are stored
    output_dir="../ml/artifacts/distilbert_checkpoints",

    # Fine-tuning learning rate
    learning_rate=2e-5,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Number of training epochs
    num_train_epochs=3,

    # Regularization
    weight_decay=0.01,

    # 10% of total training steps used for warmup
    warmup_steps=0.10,

    # Evaluate after every epoch
    eval_strategy="epoch",

    # Save after every epoch
    save_strategy="epoch",

    # Keep only two checkpoints
    save_total_limit=2,

    # Automatically restore best checkpoint
    load_best_model_at_end=True,

    # Select best model using validation Macro F1
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Training logs
    logging_strategy="steps",
    logging_steps=50,

    # No WandB / external tracker for now
    report_to="none",

    # Better compatibility with macOS/MPS
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    # Reproducibility
    seed=42
)

print("Training configuration created successfully!")

Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Confirm the device

In [ ]:
print(
    "Trainer device:",
    trainer.args.device
)

Start fine tuning

In [ ]:
train_result = trainer.train()

Evaluate the best validation model

In [ ]:
validation_results = trainer.evaluate(
    val_dataset
)

validation_results

In [ ]:
print("VALIDATION RESULTS")
print("=" * 50)

print(
    f"Accuracy: "
    f"{validation_results['eval_accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{validation_results['eval_precision_macro']:.4f}"
)

print(
    f"Macro Recall: "
    f"{validation_results['eval_recall_macro']:.4f}"
)

print(
    f"Macro F1: "
    f"{validation_results['eval_f1_macro']:.4f}"
)

print(
    f"Loss: "
    f"{validation_results['eval_loss']:.4f}"
)

Inspect the training history

In [ ]:
log_history = pd.DataFrame(
    trainer.state.log_history
)

log_history.tail(20)

Plot training loss

In [ ]:
training_logs = log_history[
    log_history["loss"].notna()
].copy()

plt.figure(figsize=(10, 6))

plt.plot(
    training_logs["step"],
    training_logs["loss"]
)

plt.title(
    "DistilBERT Training Loss"
)

plt.xlabel("Training Step")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

Plot validation Macro F1

In [ ]:
evaluation_logs = log_history[
    log_history["eval_f1_macro"].notna()
].copy()

plt.figure(figsize=(8, 5))

plt.plot(
    evaluation_logs["epoch"],
    evaluation_logs["eval_f1_macro"],
    marker="o"
)

plt.title(
    "DistilBERT Validation Macro F1"
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")

plt.xticks(
    evaluation_logs["epoch"]
)

plt.tight_layout()
plt.show()

Plot validation loss

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    evaluation_logs["epoch"],
    evaluation_logs["eval_loss"],
    marker="o"
)

plt.title(
    "DistilBERT Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")

plt.xticks(
    evaluation_logs["epoch"]
)

plt.tight_layout()
plt.show()

Check which checkpoint won

In [ ]:
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation metric:",
    trainer.state.best_metric
)

##Final DistilBERT Test Evaluation
##Evaluate the official BANKING77 test set

In [ ]:
official_test_output = trainer.predict(
    test_dataset
)

official_logits = official_test_output.predictions
official_labels = official_test_output.label_ids

official_predictions = np.argmax(
    official_logits,
    axis=-1
)

In [ ]:
distilbert_official_accuracy = accuracy_score(
    official_labels,
    official_predictions
)

distilbert_official_precision = precision_score(
    official_labels,
    official_predictions,
    average="macro",
    zero_division=0
)

distilbert_official_recall = recall_score(
    official_labels,
    official_predictions,
    average="macro",
    zero_division=0
)

distilbert_official_f1 = f1_score(
    official_labels,
    official_predictions,
    average="macro",
    zero_division=0
)

print("DISTILBERT — OFFICIAL BANKING77 TEST")
print("=" * 55)

print(
    f"Accuracy:        {distilbert_official_accuracy:.4f}"
)

print(
    f"Macro Precision: {distilbert_official_precision:.4f}"
)

print(
    f"Macro Recall:    {distilbert_official_recall:.4f}"
)

print(
    f"Macro F1:        {distilbert_official_f1:.4f}"
)

Recreate the strict test set

In [ ]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )

In [ ]:
train_normalized_texts = set(
    train_df["text"].apply(normalize_text)
)

strict_test_data = test_df[
    ~test_df["text"]
    .apply(normalize_text)
    .isin(train_normalized_texts)
].copy()

print("Official test size:", len(test_df))
print("Strict test size:", len(strict_test_data))

In [ ]:
strict_test_data["label"] = (
    strict_test_data["intent"]
    .map(label2id)
)

Create the strict PyTorch dataset

In [ ]:
strict_test_dataset = Banking77Dataset(
    strict_test_data["text"],
    strict_test_data["label"],
    tokenizer,
    MAX_LENGTH
)

In [ ]:
print(
    "Strict test examples:",
    len(strict_test_dataset)
)

Evaluate DistilBERT on strict test data

In [ ]:
strict_test_output = trainer.predict(
    strict_test_dataset
)

strict_logits = (
    strict_test_output.predictions
)

strict_labels = (
    strict_test_output.label_ids
)

strict_predictions = np.argmax(
    strict_logits,
    axis=-1
)

In [ ]:
distilbert_strict_accuracy = accuracy_score(
    strict_labels,
    strict_predictions
)

distilbert_strict_precision = precision_score(
    strict_labels,
    strict_predictions,
    average="macro",
    zero_division=0
)

distilbert_strict_recall = recall_score(
    strict_labels,
    strict_predictions,
    average="macro",
    zero_division=0
)

distilbert_strict_f1 = f1_score(
    strict_labels,
    strict_predictions,
    average="macro",
    zero_division=0
)

print("DISTILBERT — STRICT ZERO-OVERLAP TEST")
print("=" * 55)

print(
    f"Accuracy:        {distilbert_strict_accuracy:.4f}"
)

print(
    f"Macro Precision: {distilbert_strict_precision:.4f}"
)

print(
    f"Macro Recall:    {distilbert_strict_recall:.4f}"
)

print(
    f"Macro F1:        {distilbert_strict_f1:.4f}"
)

Compare DistilBERT against TF-IDF

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "DistilBERT"
    ],

    "Official Accuracy": [
        0.8588,
        distilbert_official_accuracy
    ],

    "Official Macro F1": [
        0.8581,
        distilbert_official_f1
    ],

    "Strict Accuracy": [
        0.8584,
        distilbert_strict_accuracy
    ],

    "Strict Macro F1": [
        0.8576,
        distilbert_strict_f1
    ]
})

model_comparison

Error analysis for DistilBERT

In [ ]:
actual_intents = [
    id2label[int(label)]
    for label in official_labels
]

predicted_intents = [
    id2label[int(pred)]
    for pred in official_predictions
]

In [ ]:
distilbert_results_df = pd.DataFrame({
    "text": test_data["text"].values,
    "actual": actual_intents,
    "predicted": predicted_intents
})

In [ ]:
distilbert_errors = (
    distilbert_results_df[
        distilbert_results_df["actual"]
        != distilbert_results_df["predicted"]
    ]
    .copy()
)

print(
    "Correct predictions:",
    len(distilbert_results_df)
    - len(distilbert_errors)
)

print(
    "Incorrect predictions:",
    len(distilbert_errors)
)

Find Transformer confusion pairs

In [ ]:
distilbert_confusions = (
    distilbert_errors
    .groupby(
        ["actual", "predicted"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

distilbert_confusions.head(20)

Inspect wrong predictions manually

In [ ]:
for _, row in (
    distilbert_errors.head(10).iterrows()
):

    print("=" * 80)

    print("CUSTOMER QUERY:")
    print(row["text"])

    print("\nACTUAL:")
    print(row["actual"])

    print("\nPREDICTED:")
    print(row["predicted"])

    print()

Find per-intent performance

In [ ]:
distilbert_report = classification_report(
    actual_intents,
    predicted_intents,
    output_dict=True,
    zero_division=0
)

distilbert_report_df = (
    pd.DataFrame(
        distilbert_report
    )
    .transpose()
)

In [ ]:
#isolate only the 77 intents
intent_report = (
    distilbert_report_df.loc[
        intent_names
    ]
)

In [ ]:
#worst 10
intent_report.sort_values(
    "f1-score"
).head(10)[
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

In [ ]:
#best 10
intent_report.sort_values(
    "f1-score",
    ascending=False
).head(10)[
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

reload a fresh pretrained model

In [ ]:
model_5epoch = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=77,
    id2label=id2label,
    label2id=label2id
)

model_5epoch = model_5epoch.to(device)

print("Fresh DistilBERT loaded for Experiment 2")

Create separate training arguments

In [ ]:
training_args_5epoch = TrainingArguments(

    output_dir="../ml/artifacts/distilbert_5epoch_checkpoints",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Only experimental change
    num_train_epochs=5,

    weight_decay=0.01,

    # Use the same warmup setting that worked
    # in your Experiment 1
    warmup_steps=0.10,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=50,

    report_to="none",

    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    seed=42
)

Create a new Trainer

In [ ]:
trainer_5epoch = Trainer(
    model=model_5epoch,
    args=training_args_5epoch,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

print("5-epoch Trainer created")
print("Device:", trainer_5epoch.args.device)

Start training now with 5 epochs


In [ ]:
train_result_5epoch = trainer_5epoch.train()

Check the winning checkpoint

In [ ]:
print(
    "Best checkpoint:",
    trainer_5epoch.state.best_model_checkpoint
)

print(
    "Best validation Macro F1:",
    trainer_5epoch.state.best_metric
)

In [ ]:
experiment_comparison = pd.DataFrame({
    "Experiment": [
        "DistilBERT - 3 epochs",
        "DistilBERT - 5 epochs"
    ],
    "Best Validation Macro F1": [
        0.812861492067468,
        trainer_5epoch.state.best_metric
    ]
})

experiment_comparison

Official BANKING77 test

In [ ]:
official_output_5epoch = trainer_5epoch.predict(
    test_dataset
)

official_logits_5epoch = (
    official_output_5epoch.predictions
)

official_labels_5epoch = (
    official_output_5epoch.label_ids
)

official_pred_5epoch = np.argmax(
    official_logits_5epoch,
    axis=-1
)

In [ ]:
distilbert5_official_accuracy = accuracy_score(
    official_labels_5epoch,
    official_pred_5epoch
)

distilbert5_official_precision = precision_score(
    official_labels_5epoch,
    official_pred_5epoch,
    average="macro",
    zero_division=0
)

distilbert5_official_recall = recall_score(
    official_labels_5epoch,
    official_pred_5epoch,
    average="macro",
    zero_division=0
)

distilbert5_official_f1 = f1_score(
    official_labels_5epoch,
    official_pred_5epoch,
    average="macro",
    zero_division=0
)

print("DISTILBERT 5-EPOCH — OFFICIAL TEST")
print("=" * 55)

print(
    f"Accuracy:        {distilbert5_official_accuracy:.4f}"
)

print(
    f"Macro Precision: {distilbert5_official_precision:.4f}"
)

print(
    f"Macro Recall:    {distilbert5_official_recall:.4f}"
)

print(
    f"Macro F1:        {distilbert5_official_f1:.4f}"
)

Strict zero-overlap test

In [ ]:
strict_output_5epoch = trainer_5epoch.predict(
    strict_test_dataset
)

strict_logits_5epoch = (
    strict_output_5epoch.predictions
)

strict_labels_5epoch = (
    strict_output_5epoch.label_ids
)

strict_pred_5epoch = np.argmax(
    strict_logits_5epoch,
    axis=-1
)

In [ ]:
distilbert5_strict_accuracy = accuracy_score(
    strict_labels_5epoch,
    strict_pred_5epoch
)

distilbert5_strict_precision = precision_score(
    strict_labels_5epoch,
    strict_pred_5epoch,
    average="macro",
    zero_division=0
)

distilbert5_strict_recall = recall_score(
    strict_labels_5epoch,
    strict_pred_5epoch,
    average="macro",
    zero_division=0
)

distilbert5_strict_f1 = f1_score(
    strict_labels_5epoch,
    strict_pred_5epoch,
    average="macro",
    zero_division=0
)

print("DISTILBERT 5-EPOCH — STRICT TEST")
print("=" * 55)

print(
    f"Accuracy:        {distilbert5_strict_accuracy:.4f}"
)

print(
    f"Macro Precision: {distilbert5_strict_precision:.4f}"
)

print(
    f"Macro Recall:    {distilbert5_strict_recall:.4f}"
)

print(
    f"Macro F1:        {distilbert5_strict_f1:.4f}"
)

create the important model comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "DistilBERT - 3 epochs",
        "DistilBERT - 5 epochs"
    ],

    "Official Accuracy": [
        0.8588,
        0.8159,
        distilbert5_official_accuracy
    ],

    "Official Macro F1": [
        0.8581,
        0.7975,
        distilbert5_official_f1
    ],

    "Strict Accuracy": [
        0.8584,
        0.8161,
        distilbert5_strict_accuracy
    ],

    "Strict Macro F1": [
        0.8576,
        0.7975,
        distilbert5_strict_f1
    ]
})

comparison_df

Final Error Analysis

In [ ]:
actual_intents_5epoch = [
    id2label[int(label)]
    for label in official_labels_5epoch
]

predicted_intents_5epoch = [
    id2label[int(pred)]
    for pred in official_pred_5epoch
]

distilbert5_results = pd.DataFrame({
    "text": test_data["text"].values,
    "actual": actual_intents_5epoch,
    "predicted": predicted_intents_5epoch
})

distilbert5_errors = distilbert5_results[
    distilbert5_results["actual"]
    != distilbert5_results["predicted"]
].copy()

print(
    "Total test examples:",
    len(distilbert5_results)
)

print(
    "Correct predictions:",
    len(distilbert5_results)
    - len(distilbert5_errors)
)

print(
    "Incorrect predictions:",
    len(distilbert5_errors)
)

print(
    "Error rate:",
    round(
        len(distilbert5_errors)
        / len(distilbert5_results)
        * 100,
        2
    ),
    "%"
)

In [ ]:
confusion_pairs_5epoch = (
    distilbert5_errors
    .groupby(
        ["actual", "predicted"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)

confusion_pairs_5epoch.head(20)

inspect some actual mistakes

In [ ]:
for _, row in (
    distilbert5_errors.head(15).iterrows()
):

    print("=" * 80)

    print("CUSTOMER QUERY:")
    print(row["text"])

    print("\nACTUAL INTENT:")
    print(row["actual"])

    print("\nPREDICTED INTENT:")
    print(row["predicted"])

    print()

Per-intent performance

In [ ]:
distilbert5_report = classification_report(
    actual_intents_5epoch,
    predicted_intents_5epoch,
    output_dict=True,
    zero_division=0
)

distilbert5_report_df = (
    pd.DataFrame(
        distilbert5_report
    )
    .transpose()
)

In [ ]:
intent_performance = (
    distilbert5_report_df.loc[
        intent_names
    ]
)

Worst 10 intents

In [ ]:
worst_10_intents = (
    intent_performance
    .sort_values("f1-score")
    .head(10)
)

worst_10_intents[
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

Best 10 intents

In [ ]:
best_10_intents = (
    intent_performance
    .sort_values(
        "f1-score",
        ascending=False
    )
    .head(10)
)

best_10_intents[
    [
        "precision",
        "recall",
        "f1-score",
        "support"
    ]
]

Save the final Transformer model

In [ ]:
from pathlib import Path

FINAL_MODEL_DIR = Path(
    "../ml/artifacts/banking77_distilbert_final"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
trainer_5epoch.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print(
    "Final DistilBERT model saved to:",
    FINAL_MODEL_DIR
)

Save evaluation results

In [ ]:
EVALUATION_DIR = Path(
    "../ml/evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
#save comparisons
comparison_df.to_csv(
    EVALUATION_DIR
    / "intent_model_comparison.csv",
    index=False
)

In [ ]:
#save confusion paris
confusion_pairs_5epoch.to_csv(
    EVALUATION_DIR
    / "distilbert_confusion_pairs.csv",
    index=False
)

In [ ]:
#save per-intent metrics
intent_performance.to_csv(
    EVALUATION_DIR
    / "distilbert_per_intent_metrics.csv"
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

tokenizer.save_pretrained(
    "../ml/artifacts/banking77_distilbert_final"
)

print("Tokenizer saved successfully!")